# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals_common import DealSelection
from agents.deals import ScrapedDeal


In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [24:55<00:00, 166.21s/it]


In [4]:
len(deals)

90

In [5]:
deals[44].describe()

"Title: YETI 10 oz. Rambler Tumbler with MagSlider Lid for $15 + free shipping w/ $49\nDetails: It's a savings of $5 and the best price we could find. ScoreCard members get free shipping on orders of $49 or more. It's free to join. In-store pickup may also available. Buy Now at Dick's Sporting Goods\nFeatures: \nURL: https://www.dealnews.com/YETI-10-oz-Rambler-Tumbler-with-Mag-Slider-Lid-for-15-free-shipping-w-49/21741976.html?iref=rss-c196"

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Father's Day Gifts at JCPenney Under $15 + free shipping w/ $49
Details: Shop deals for dad at budget prices including apparel, electronics, personal care items, and more. Shipping adds $8.95, but orders of $49 or more ship for free. Pickup is also available. Buy Now at JCPenney
Features: 
URL: https://www.dealnews.com/Fathers-Day-Gifts-at-JCPenney-Under-15-free-shi

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description="The Samsung Bespoke Jet Bot Combo Robot Vacuum and Mop features advanced LiDAR navigation with 3D mapping capabilities, ensuring thorough and efficient cleaning throughout your home. Equipped with an auto-wash feature for the mop and app control, this robot gets your floors sparkling clean with minimal effort on your part. The included All-in-One Clean Station allows for hassle-free emptying and cleaning of your robot's dirt bin—ideal for maintaining a tidy home. It's particularly suitable for busy households looking for automation in their cleaning routines.", price=750.0, url='https://www.dealnews.com/products/Samsung/Samsung-Bespoke-Jet-Bot-Combo-Robot-Vacuum-Mop-w-All-in-One-Clean-Station/490288.html?iref=rss-f1912')

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

In [15]:
result.deals

[Deal(product_description='The LG S40T 2.1-Channel Soundbar with Wireless Subwoofer enhances your audio experience with Dolby Digital and DTS Digital compatibility. It features Dolby Audio Clear Voice Plus and comes with a 3-band equalizer for customized sound settings. Ideal for movie nights or music sessions, this soundbar provides immersive audio. Coupled with a 2-year Allstate warranty, it ensures reliability and quality for your listening pleasure.', price=144.0, url='https://www.dealnews.com/products/LG/LG-S40-T-2-1-Channel-Soundbar-with-Wireless-Subwoofer/483662.html?iref=rss-c142'),
 Deal(product_description='The Jackery Explorer 2000 Plus is a powerful 3,000W power station that supports heavy-duty devices up to 6000W. It utilizes LiFePO4 battery technology, ensuring a lifespan of up to 10 years. This portable power solution is equipped with ChargeShield technology, allowing for secure and efficient power management. It comes with a battery pack and a 500W solar panel, making i

In [16]:
print(result.deals[0])

product_description='The LG S40T 2.1-Channel Soundbar with Wireless Subwoofer enhances your audio experience with Dolby Digital and DTS Digital compatibility. It features Dolby Audio Clear Voice Plus and comes with a 3-band equalizer for customized sound settings. Ideal for movie nights or music sessions, this soundbar provides immersive audio. Coupled with a 2-year Allstate warranty, it ensures reliability and quality for your listening pleasure.' price=144.0 url='https://www.dealnews.com/products/LG/LG-S40-T-2-1-Channel-Soundbar-with-Wireless-Subwoofer/483662.html?iref=rss-c142'
